In [1]:
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb
import shap
import pickle
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

d:\Ciencias de Dados\Projetos\creditRisk_DecisionEngine\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
X_train = pd.read_parquet("../data/processed/X_train.parquet")
y_train = pd.read_parquet("../data/processed/y_train.parquet")

X_test = pd.read_parquet("../data/processed/X_test.parquet")
y_test = pd.read_parquet("../data/processed/y_test.parquet")

In [3]:
X_train.columns

Index(['credit_utilization', 'age', 'late_30_59_days', 'debt_ratio',
       'monthly_income', 'open_credit_lines', 'late_90_days',
       'real_estate_loans', 'late_60_89_days', 'dependents',
       'income_per_dependent', 'weighted_late'],
      dtype='str')

## BaseLine

In [4]:
baselineModel = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    verbose=0
)

baselineModel.fit(X_train, y_train)

d:\Ciencias de Dados\Projetos\creditRisk_DecisionEngine\.venv\Lib\site-packages\sklearn\utils\validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
d:\Ciencias de Dados\Projetos\creditRisk_DecisionEngine\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

## Model 

In [5]:
skf = StratifiedKFold(    n_splits=6,    shuffle=True,   random_state=42 )
lgb_model = lgb.LGBMClassifier(objective="binary",metric="auc")

In [6]:
param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'learning_rate': [0.01, 0.015, 0.02, 0.013, 0.05, 0.1],
    'num_leaves': [10,15,20,15,31, 50, 70],
    'max_depth': [-1, 3,5 ,7 , 10, 12],
    'subsample': [0.3, 0.5, 0.7, 0.8, 1.0],
    'colsample_bytree': [0.3, 0.5, 0.7, 0.8, 1.0],
    'lambda_l1' : [0.1,0.3,0.5],
    'lambda_l2' : [0.1,0.3,0.5],
    'subsample_freq': [0, 1, 5],
    'min_child_samples': [10, 20, 30, 50, 100]
}

In [7]:
random_search = RandomizedSearchCV(
    estimator=lgb_model,    param_distributions=param_dist,    n_iter=100,    scoring='roc_auc',
    cv=skf,    verbose=0,    random_state=42,    n_jobs=-1
)

random_search.fit(X_train, y_train)

KeyboardInterrupt: 

In [ ]:
final_model = random_search.best_estimator_
final_model.fit(X_train, y_train)

d:\Ciencias de Dados\Projetos\creditRisk_DecisionEngine\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:103: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
d:\Ciencias de Dados\Projetos\creditRisk_DecisionEngine\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


[LightGBM] [Warning] lambda_l1 is set=0.5, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.5
[LightGBM] [Warning] lambda_l2 is set=0.5, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.5
[LightGBM] [Warning] lambda_l1 is set=0.5, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.5
[LightGBM] [Warning] lambda_l2 is set=0.5, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.5
[LightGBM] [Info] Number of positive: 6717, number of negative: 93783
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001691 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1308
[LightGBM] [Info] Number of data points in the train set: 100500, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.066836 -> initscore=-2.636342
[LightGBM] [Info] Start training from score -2.636342


,boosting_type,'gbdt'
,num_leaves,15
,max_depth,10
,learning_rate,0.015
,n_estimators,500
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,30


## Evaluate

In [ ]:
baselinePreds = baselineModel.predict(X_test)
finalModPreds = final_model.predict_proba(X_test)

[LightGBM] [Warning] lambda_l1 is set=0.5, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.5
[LightGBM] [Warning] lambda_l2 is set=0.5, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.5


In [ ]:
baselineAuc = roc_auc_score(y_test, baselinePreds)
finalModAuc = roc_auc_score(y_test, finalModPreds[:,1])
print("BaseLine ROC AUC:", baselineAuc)
print("FinalModel ROC AUC:", finalModAuc)

BaseLine ROC AUC: 0.7780129523687496
FinalModel ROC AUC: 0.8669193111271023


In [ ]:
#explainer = shap.Explainer(final_model, X_train)
#shap_values = explainer(X_test)

In [ ]:
#shap.plots.bar(shap_values) 

In [ ]:
#shap.plots.beeswarm(shap_values)

In [ ]:
#shap.plots.beeswarm(shap_values)

In [ ]:
threshold_2 = .2
threshold_25 = .25
threshold_3 = .3
threshold_5 = .5
threshold_7 = .7

In [ ]:
#realResult = (y_test >= threshold).astype(int)
#finalModResult = (finalModPreds[:,1] >= threshold).astype(int)

In [ ]:
X_test["Result"] = finalModPreds[:,1]
X_test["Target"] = y_test["Target"]

In [ ]:
#with open("../data/final/final_model.pkl", "wb") as f:
#    pickle.dump(final_model, f)
#
##with open("../data/final/explainer.pkl", "wb") as f:
##    pickle.dump(explainer, f)
#
#X_test.to_parquet("../data/final/X_test.parquet")
#
#y_test[["Target"]].to_parquet("../data/final/y_test.parquet")
#y_test[["y_pred"]].to_parquet("../data/final/preds.parquet")

KeyError: "None of [Index(['y_pred'], dtype='str')] are in the [columns]"

In [ ]:
y_test[["Target"]]

,Target
118607,0
112706,0
57891,0
13081,0
141317,0
...,...
103450,0
54368,1
144666,0
97933,0


In [ ]:
y_test["Result"] = y_test["y_pred"]>= threshold
y_test["Result"] = y_test["Result"].astype(int)

NameError: name 'y_test' is not defined

In [ ]:

print(classification_report(y_test["Target"], y_test["Result"]))

NameError: name 'y_test' is not defined

In [ ]:
y_test

,Target,Result
118607,0,0.064701
112706,0,0.036705
57891,0,0.021328
13081,0,0.045992
141317,0,0.007298
...,...,...
103450,0,0.046124
54368,1,0.715783
144666,0,0.013763
97933,0,0.017680


,credit_utilization,age,late_30_59_days,debt_ratio,monthly_income,open_credit_lines,late_90_days,real_estate_loans,late_60_89_days,dependents,income_per_dependent,weighted_late
118607,0.336032,55,0,0.938677,6000.00,11,0,3,0,1.0,3000.00,0
112706,0.502591,35,0,0.045514,7667.00,5,0,0,0,1.0,3833.50,0
57891,0.000000,29,0,0.213468,3058.00,6,0,0,0,0.0,3058.00,0
13081,0.522950,53,0,0.205122,4333.00,10,0,0,0,0.0,4333.00,0
141317,0.003100,47,0,2502.000000,0.05,3,0,1,0,0.0,0.05,0
...,...,...,...,...,...,...,...,...,...,...,...,...
103450,0.049132,35,1,20.000000,127.21,3,0,0,0,0.0,127.21,1
54368,1.092956,37,1,0.162893,3400.00,5,3,0,2,0.0,3400.00,14
144666,0.000000,40,0,3674.000000,0.00,10,0,2,0,0.0,0.00,0
97933,0.015279,49,0,0.245791,16094.00,6,0,4,0,0.0,16094.00,0
